In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("Dados/siga-empreendimentos-geracao.csv", sep=";", encoding="latin-1")
print(df.shape)
df.head()

In [ ]:
new_df = df[[
    'CodCEG', 'SigTipoGeracao', 'DscOrigemCombustivel', 'DscFonteCombustivel',
    'MdaPotenciaOutorgadaKw', 'NumCoordNEmpreendimento', 'NumCoordEEmpreendimento',
    'DatInicioVigencia', 'DatFimVigencia'
]]

new_df.head()

In [ ]:
import geopandas as gpd
import folium
from folium import FeatureGroup, LayerControl

run_again = True

if run_again:
    # Converter coordenadas para GeoDataFrame
    # E = Longitude (X), N = Latitude (Y)
    new_df = new_df.copy()
    new_df['lon'] = new_df['NumCoordEEmpreendimento'].astype(str).str.replace(',', '.').astype(float)
    new_df['lat'] = new_df['NumCoordNEmpreendimento'].astype(str).str.replace(',', '.').astype(float)

    # Remover linhas sem coordenadas válidas
    new_df = new_df.dropna(subset=['lat', 'lon'])
    new_df = new_df[(new_df['lat'].between(-90, 90)) & (new_df['lon'].between(-180, 180))]

    gdf = gpd.GeoDataFrame(
        new_df,
        geometry=gpd.points_from_xy(new_df['lon'], new_df['lat']),
        crs='EPSG:4326'
    )

    # Cores e nomes para cada tipo de geração
    tipos_geracao = {
        'UTN': ('red',    'Usina Termonuclear'),
        'UTE': ('orange', 'Usina Termelétrica'),
        'UHE': ('blue',   'Usina Hidrelétrica'),
        'UFV': ('gold',   'Central Geradora Solar Fotovoltaica'),
        'PCH': ('cyan',   'Pequena Central Hidrelétrica'),
        'EOL': ('green',  'Central Geradora Eólica'),
        'CGU': ('purple', 'Central Geradora Undi-elétrica'),
        'CGH': ('brown',  'Central Geradora Hidrelétrica'),
    }

    # Criar mapa base centrado no Brasil
    m = folium.Map(location=[-10, -55], zoom_start=4, tiles='OpenStreetMap')

    # Plotar cada tipo de geração em seu próprio FeatureGroup (toggle na legenda)
    for tipo_geracao, (cor, nome_oficial) in tipos_geracao.items():
        gdf_tipo = gdf[gdf['SigTipoGeracao'] == tipo_geracao]

        if gdf_tipo.empty:
            continue

        fg = FeatureGroup(name=nome_oficial, show=True)

        for _, row in gdf_tipo.iterrows():
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=5,
                popup=folium.Popup(
                    f"<b>{nome_oficial}</b><br>Potência: {row['MdaPotenciaOutorgadaKw']} kW",
                    max_width=200
                ),
                tooltip=nome_oficial,
                color=cor,
                fill=True,
                fill_color=cor,
                fill_opacity=0.7,
                weight=1,
            ).add_to(fg)

        fg.add_to(m)

    # Adicionar controle de camadas (toggle por tipo)
    LayerControl(collapsed=False).add_to(m)

    # Adicionar legenda HTML com swatches visíveis
    legend_html = '''
    <div style="
        position: fixed;
        bottom: 50px; right: 50px;
        width: 290px;
        background-color: white;
        border: 2px solid #666;
        border-radius: 6px;
        z-index: 9999;
        font-size: 13px;
        font-family: Arial, sans-serif;
        padding: 12px 14px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    ">
    <p style="margin:0 0 8px 0; font-weight:bold; font-size:14px;">Tipos de Geração</p>
    '''
    for tipo_geracao, (cor, nome_oficial) in tipos_geracao.items():
        legend_html += (
            f'<div style="display:flex; align-items:center; margin:4px 0;">'
            f'<span style="display:inline-block; width:14px; height:14px; '
            f'background-color:{cor}; border-radius:50%; margin-right:8px; flex-shrink:0;"></span>'
            f'{nome_oficial}</div>'
        )
    legend_html += '</div>'

    m.get_root().html.add_child(folium.Element(legend_html))

    m.save('mapa_geracoes.html')
    m